# macrocast — Quick Start

This notebook demonstrates the full macrocast pipeline on synthetic data.
No internet connection required.

**What we cover:**
- Constructing a `MacroFrame` from scratch
- Running a `ForecastExperiment` with one model (KRR) at `h=1`
- Inspecting the `ResultSet` and computing MSFE / OOS-R²
- Running the Diebold-Mariano test

In [ ]:
import numpy as np
import pandas as pd

from macrocast.data.schema import MacroFrame, MacroFrameMetadata, VariableMetadata
from macrocast.pipeline.components import (
    CVScheme,
    LossFunction,
    Nonlinearity,
    Regularization,
    Window,
)
from macrocast.pipeline.experiment import FeatureSpec, ForecastExperiment, ModelSpec
from macrocast.pipeline.models import KRRModel
from macrocast.evaluation.metrics import msfe, relative_msfe, oos_r2
from macrocast.evaluation.dm import dm_test

## 1. Synthetic data

In [ ]:
rng = np.random.default_rng(42)

T, N = 200, 30
dates = pd.date_range("1990-01", periods=T, freq="MS")

# Latent factor drives both the target and some predictors
factor = rng.standard_normal(T)
X_raw = np.column_stack([
    factor[:, None] * rng.standard_normal((1, N // 2)) + rng.standard_normal((T, N // 2)),
    rng.standard_normal((T, N - N // 2)),
])
y_raw = factor * 0.8 + rng.standard_normal(T) * 0.3

X_df = pd.DataFrame(X_raw, index=dates, columns=[f"X{i:02d}" for i in range(N)])
y_df = pd.DataFrame({"TARGET": y_raw}, index=dates)

print(f"Panel shape: {X_df.shape}")
print(f"Target shape: {y_df.shape}")

In [ ]:
# Build MacroFrame
var_meta = [
    VariableMetadata(name=c, tcode=1, group="other", is_target=False)
    for c in X_df.columns
]
var_meta.append(
    VariableMetadata(name="TARGET", tcode=1, group="other", is_target=True)
)

meta = MacroFrameMetadata(
    frequency="M",
    vintage="synthetic",
    source="synthetic",
    variables=var_meta,
)

full_df = pd.concat([X_df, y_df], axis=1)
mf = MacroFrame(data=full_df, metadata=meta)
print(mf)

## 2. Experiment setup

In [ ]:
# KRR with RBF kernel, K-fold CV, MSE loss
models = [
    ModelSpec(
        model_cls=KRRModel,
        nonlinearity=Nonlinearity.KRR,
        regularization=Regularization.RIDGE,
        cv_scheme=CVScheme.kfold(k=5),
        loss_function=LossFunction.L2,
        model_kwargs={"cv_folds": 5},
    ),
]

feat_spec = FeatureSpec(
    n_factors=4,
    n_lags=4,
    use_factors=True,
)

exp = ForecastExperiment(
    macro_frame=mf,
    target_col="TARGET",
    horizons=[1],
    model_specs=models,
    feature_spec=feat_spec,
    window=Window.EXPANDING,
    min_train_size=60,
    n_jobs=1,
)

print("Experiment configured.")

## 3. Run

In [ ]:
result_set = exp.run()
df = result_set.to_dataframe()
print(f"Result rows: {len(df)}")
df.head()

## 4. Evaluation

In [ ]:
model_id = df["model_id"].iloc[0]
sub = df[df["model_id"] == model_id]

y_true = sub["y_true"].values
y_hat  = sub["y_hat"].values

# Naive benchmark: predict the in-sample mean
bench_hat = np.full_like(y_true, y_true.mean())

print(f"MSFE (KRR):      {msfe(y_true, y_hat):.4f}")
print(f"MSFE (benchmark):{msfe(y_true, bench_hat):.4f}")
print(f"Relative MSFE:   {relative_msfe(y_true, y_hat, bench_hat):.4f}")
print(f"OOS R²:          {oos_r2(y_true, y_hat, bench_hat):.4f}")

In [ ]:
# Diebold-Mariano test: KRR vs naive benchmark
dm = dm_test(y_true, y_hat, bench_hat, h=1, loss="mse", hln_adjust=True)
print(f"DM statistic: {dm.dm_stat:.3f}")
print(f"p-value:      {dm.p_value:.4f}")
print(f"Mean loss diff (KRR - bench): {dm.loss_diff_mean:.4f}")

## 5. MSFE summary by model

In [ ]:
summary = result_set.msfe_by_model()
print(summary)